# Planck · справочный ноутбук HEALPix

Исходный учебный материал по данным. Загрузки направлены в `data/raw/`; обработанные изображения находятся в `data/processed/`. В отличие от итогового анализа этот ноутбук требует внешних данных и ручного выбора примеров. [Воспроизведение](../../docs/reproduction.md).


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
    and (path / "src" / "galaxy_mass_estimation").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

[Сайт](https://wiki.cosmos.esa.int/planck-legacy-archive/index.php/Planck_Collaboration) от коллаборации о данных и миссии

Как работают с картами Planck c точки зрения DL [github](https://github.com/rt2122/AstroDeepLearning)

In [ ]:
(PROJECT_ROOT / "data/raw").mkdir(parents=True, exist_ok=True)

Мы используем карты [y-параметра](https://ru.wikipedia.org/wiki/Эффект_Сюняева_—_Зельдовича). 
y - параметр характеризует насколько скопление исказило чернотельный спектр реликтового излучения

$$y = \int \frac{k T_e}{m c^2} n_e \sigma_t dl$$

где $n_e$ и $T_e$ - концентрация и температура электронов, $\sigma_t$ - томпсоновское рассеяние 

Данные с Planck берем [отсюда](https://zenodo.org/records/7947597)

Об алгоритме [NILC](https://wiki.cosmos.esa.int/planckpla2015/index.php/Astrophysical_component_separation), с помощью которого подготовлены данные 

Данные хранятся в формате healpy 

# Download data

In [ ]:
!wget -P "{PROJECT_ROOT / 'data/raw'}" https://zenodo.org/records/7947597/files/PR4_NILC_y_map.fits

# Import packages

In [ ]:
!pip install healpy

In [ ]:
import healpy as hp, numpy as np, matplotlib.pyplot as plt

# Healpy library

Разрешение карты определяется параметром NSIDE, который, как правило, выражается степенью $2$.

In [ ]:
NSIDE = 2
print(
    "Угловое разрешение NSIDE {} примерно {:.2} deg".format(
        NSIDE, hp.nside2resol(NSIDE, arcmin=True) / 60
    )
)

Функция `hp.pixelfunc.nside2npix` выдаёт число пикселей карты `NPIX` по формуле $12\times$ `NSIDE`$^2$:

In [ ]:
NPIX = hp.nside2npix(NSIDE)
NPIX, 12 * NSIDE**2

Доступно два способа упорядочивания пикселей: `RING` (в виде колец, начиная с северного полюса налево в сторону востока, задаётся nest = **False**) и `NESTED` (nest = **True**).

In [ ]:
f, ax = plt.subplots(4, 3)

for i in range(0, 12):
    m = np.zeros(NPIX)
    m[i] = 1
    plt.axes(ax[i // 3, i % 3])
    hp.mollview(m, nest=False, title="m = {}".format(i), hold=True)

In [ ]:
f, ax = plt.subplots(4, 3)

for i in range(0, 12):
    m = np.zeros(NPIX)
    m[i] = 1
    plt.axes(ax[i // 3, i % 3])
    hp.mollview(m, nest=True, title="m = {}".format(i), hold=True)

In [ ]:
m = np.arange(NPIX)
m[0] = 100
m[1] = 200
m[2] = 300
m[3] = 400
m[4] = 100  # лежит уже в другом "гнезде"
hp.mollview(m, nest=True, title="HEALPix map (NESTED ordering)")
hp.graticule()  # turns on/off the grid

Стандартные координаты --- коширота $\theta$ (**colatitude**)  с $0$ на северном полюсе, $\pi/2$ на экваторе и $\pi$ на южном полюсе и долгота $\phi$  (**longitude**) от $0$ до $2\pi$, где $0$ в центре, обход в левую сторону.

Угловые координаты можно задавать векторами, используя функцию `hp.ang2vec`.

Так, например, направление $\theta=\frac{\pi}{2}, \phi=\frac{3\pi}{4}$ задаётся нормализованным вектором:

In [ ]:
vec = hp.ang2vec(np.pi / 2, np.pi * 3 / 4)
print(vec), print(sum(vec[i] ** 2 for i in range(3)) ** 0.5)

Также можно вызвать индексы пикселей внутри заданного углового радиуса.

Значения в этих индексах впоследствии удобно изменять.

Параметры `nest` должны быть согласованы в обоих функциях.

In [ ]:
ipix_disc = hp.query_disc(nside=16, nest=False, vec=vec, radius=np.radians(25))
ipix_disc

In [ ]:
m = np.arange(12 * 16**2)
m[ipix_disc] = 0
hp.mollview(m, nest=False, title="HEALPix map (RING ordering)", fig=1)

Кошироту и долготу пикселя можно узнать функцией `pix2ang`.

In [ ]:
theta, phi = np.degrees(hp.pix2ang(nside=32, ipix=[0, 1, 2, 3, 4]))

In [ ]:
theta

In [ ]:
phi

По умолчанию карты конвертируются в *RING*, даже если они были упорядочены в формате *NESTED*. `nest=None` оставляет упорядочивание неизмененным.

Считывание карты команды `read_map` загружает по умолчанию первую колонку, для чтения данных из остальных колонок неоюходимо прописывать `field`.

`write_map` записывает карту в FITS формате.
Если карта представляет собой список из трёх различных карт, то они записываются в один файл как I,Q,U компоненты поляризации (параметры Стокса):


# Plot map

In [ ]:
ymap = hp.read_map(PROJECT_ROOT / "data/raw/PR4_NILC_y_map.fits", field=(0), h=False)

Главное свойство реликтового фона - он однороден, поэтому если мы просто нарисуем карту, то мы ничего не увидим

In [ ]:
hp.mollview(ymap, title="ymap", fig=1)

Но если обрезать по перцентилям

In [ ]:
hp.mollview(
    ymap, min=np.percentile(ymap, 5), max=np.percentile(ymap, 95), title="ymap", fig=1
)

Построим кусок карты в проекции - Gnomonic Projection, Galactic & Celestial

In [ ]:
ra_center, dec_center = 40, 4
hp.gnomview(
    ymap,
    rot=(ra_center, dec_center),
    coord=("G", "C"),
    xsize=40,
    reso=1.7,
    cmap="viridis",
    return_projected_map=False,
)
plt.show()